# Lista de Exercicios I - Programacao Funcional

**Disciplina:** Linguagens de Programacao 2026-1  
**Alunos:** Adrian Batista Pereira, Miguel Oliveira Moraes de Souza e Pedro Henrique Belota Gadelha

Este notebook reune as respostas conceituais, a implementacao funcional do Farkle em Python, a traducao para Kotlin, os testes de validacao e a comparacao entre Python e Kotlin.

## Parte I - Conceitos de Programacao Funcional

**Funcoes anonimas:** sao funcoes sem nome, normalmente usadas quando a operacao e curta e passada diretamente como argumento para outra funcao. Em Python, aparecem com `lambda`; em Kotlin, podem ser escritas como expressoes lambda entre chaves.

**Funcoes de primeira ordem:** sao funcoes tratadas como valores. Elas podem ser guardadas em variaveis, passadas como parametros e retornadas por outras funcoes.

**Funcoes de alta ordem:** sao funcoes que recebem outras funcoes como argumento ou retornam funcoes como resultado. Exemplos comuns sao `map`, `filter`, `reduce` e `fold`.

**Funcoes recursivas:** sao funcoes que chamam a si mesmas para resolver um problema menor ate chegar a um caso base. Elas substituem lacos imperativos em muitos programas funcionais.

**Funcoes puras:** sao funcoes cujo resultado depende apenas dos parametros e que nao produzem efeitos colaterais, como alterar variaveis externas, modificar listas existentes ou imprimir na tela.

**Avaliacao preguicosa:** e a estrategia em que uma expressao so e calculada quando seu valor e realmente necessario. Isso permite trabalhar com sequencias potencialmente grandes e evitar computacoes desnecessarias.

**Monads:** sao estruturas que encapsulam valores junto com um contexto computacional, como possibilidade de erro, ausencia de valor, estado ou entrada/saida. Elas permitem compor operacoes mantendo esse contexto de forma controlada.

**Estruturas de dados recursivas:** sao estruturas definidas em termos delas mesmas, como listas ligadas e arvores. Uma lista, por exemplo, pode ser vista como vazia ou como um elemento seguido por outra lista.

## Parte II - Farkle em Python Funcional

A implementacao abaixo evita `for` e `while` imperativos. Quando ha repeticao, usa recursao, list comprehensions, `map` e `reduce`. As funcoes de pontuacao sao puras; as funcoes que rolam dados e simulam partidas dependem de aleatoriedade por causa das regras do jogo.

In [ ]:
import random
from functools import reduce


def rolar_dados(n):
    """
    Rola n dados e retorna uma lista com valores de 1 a 6.
    Aqui usamos list comprehension, como pedido no trabalho.
    """
    return [random.randint(1, 6) for _ in range(n)]


def contar(lst, alvo):
    """
    Conta quantas vezes um valor aparece na lista.
    A função foi feita com recursão, sem usar for ou while.
    """
    if not lst:
        return 0

    primeiro = 1 if lst[0] == alvo else 0
    return primeiro + contar(lst[1:], alvo)


def valores_unicos(lst):
    """
    Retorna os valores únicos da lista.
    A ordem considerada é a ordem em que os elementos aparecem.
    """
    if not lst:
        return []

    primeiro = lst[0]
    resto = valores_unicos(lst[1:])

    if primeiro in resto:
        return resto

    return [primeiro] + resto


def valores_unicos_(lst):
    """Alias com o nome usado no enunciado."""
    return valores_unicos(lst)


def eh_sequencia(dados):
    """
    Verifica se os dados formam a sequência 1, 2, 3, 4, 5, 6.
    Essa jogada vale 1500 pontos.
    """
    return sorted(dados) == [1, 2, 3, 4, 5, 6]


def pontos_por_numero(dados, num):
    """
    Calcula a pontuação referente a um número específico.

    Exemplo:
    - três dados com valor 2 valem 200 pontos;
    - três dados com valor 1 valem 1000 pontos;
    - cada 1 sozinho vale 100;
    - cada 5 sozinho vale 50.
    """
    qtd = contar(dados, num)

    if qtd < 3:
        if num == 1:
            return (qtd * 100, qtd)
        if num == 5:
            return (qtd * 50, qtd)

        return (0, 0)

    trincas = qtd // 3
    resto = qtd % 3

    pontos_trinca = 1000 if num == 1 else num * 100
    pontos = trincas * pontos_trinca
    dados_usados = trincas * 3

    if num == 1:
        return (pontos + resto * 100, dados_usados + resto)

    if num == 5:
        return (pontos + resto * 50, dados_usados + resto)

    return (pontos, dados_usados)


def pontos_por_numero_(dados, num):
    """Alias com o nome usado no enunciado."""
    return pontos_por_numero(dados, num)


def pontuacao_jogada(dados):
    """
    Calcula a pontuação total de uma jogada.

    Primeiro verifica se é sequência.
    Se não for, soma a pontuação de cada número de 1 a 6.
    """
    if eh_sequencia(dados):
        return (1500, 6)

    resultados = map(
        lambda num: pontos_por_numero(dados, num),
        [1, 2, 3, 4, 5, 6]
    )

    return reduce(
        lambda total, atual: (total[0] + atual[0], total[1] + atual[1]),
        resultados,
        (0, 0)
    )


def pontuacao_jogada_(dados):
    """Alias com o nome usado no enunciado."""
    return pontuacao_jogada(dados)


def escolha_aleatoria(n_restante, pontos):
    """
    Estratégia simples: escolhe aleatoriamente se continua ou para.
    """
    return random.choice([True, False])


def escolha_heuristica(n_restante, pontos):
    """
    Estratégia criada para tentar ser melhor que a aleatória.

    Ideia:
    - se já fez pelo menos 300 pontos na rodada, para;
    - se restam poucos dados, também para;
    - caso contrário, continua.
    """
    if pontos >= 300:
        return False

    if n_restante <= 2:
        return False

    return True


def turno(pontos_rodada, n_dados, escolha, verbose=True):
    """
    Executa o turno de um jogador.

    O jogador rola os dados, ganha pontos se possível,
    e decide se continua ou para.
    """
    dados = rolar_dados(n_dados)
    pontos, dados_pontuadores = pontuacao_jogada(dados)

    if pontos == 0:
        if verbose:
            print(
                f"Continua com {dados} (0 pts) | "
                f"FARKLE! Perdeu {pontos_rodada} pontos acumulados."
            )

        return 0

    novo_total = pontos_rodada + pontos
    n_restante = n_dados - dados_pontuadores

    if n_restante == 0:
        n_restante = 6

    if verbose:
        print(
            f"Lança {dados} ({pontos} pts) | "
            f"Acumulado: {novo_total} | "
            f"Restantes: {n_restante} dados"
        )

    if escolha(n_restante, novo_total):
        return turno(novo_total, n_restante, escolha, verbose)

    if verbose:
        print(f"Jogador PARA com {novo_total} pontos")

    return novo_total


def partida(
    estrategia0,
    estrategia1,
    pontos=None,
    jogador=0,
    pontuacao_max=10000,
    verbose=False
):
    """
    Simula uma partida entre dois jogadores.

    Jogador 0 usa estrategia0.
    Jogador 1 usa estrategia1.
    """
    if pontos is None:
        pontos = [0, 0]

    if pontos[0] >= pontuacao_max or pontos[1] >= pontuacao_max:
        if verbose:
            print(
                f"\nFIM DE JOGO! "
                f"Jogador 0: {pontos[0]} | Jogador 1: {pontos[1]}"
            )

        if pontos[0] >= pontuacao_max and pontos[1] >= pontuacao_max:
            if pontos[0] > pontos[1]:
                return 0
            if pontos[1] > pontos[0]:
                return 1
            return None

        return 0 if pontos[0] >= pontuacao_max else 1

    if verbose:
        print(f"\n- Jogador {jogador} | Placar: {pontos[0]}-{pontos[1]}")

    escolha = estrategia0 if jogador == 0 else estrategia1
    pontos_do_turno = turno(0, 6, escolha, verbose)

    novos_pontos = [
        pontos[0] + (pontos_do_turno if jogador == 0 else 0),
        pontos[1] + (pontos_do_turno if jogador == 1 else 0)
    ]

    return partida(
        estrategia0,
        estrategia1,
        novos_pontos,
        1 - jogador,
        pontuacao_max,
        verbose
    )


def simular(n=1000):
    """
    Simula várias partidas e calcula o percentual de vitórias
    da estratégia heurística contra a estratégia aleatória.

    Aqui usamos reduce para evitar for/while e também evitar
    estourar o limite de recursão do Python.
    """
    vitorias = reduce(
        lambda total, _: total + (
            1 if partida(escolha_heuristica, escolha_aleatoria) == 0 else 0
        ),
        range(n),
        0
    )

    percentual = vitorias / n * 100
    return f"Heurística venceu {percentual:.2f}% das partidas contra aleatório."


def testar_pontuacao():
    """
    Testes simples para verificar se a pontuação está correta.
    Esses casos também ajudam depois na tradução para Kotlin.
    """
    casos = [
        ([1, 5, 2, 3, 4, 6], (1500, 6)),
        ([2, 2, 3, 4, 4, 6], (0, 0)),
        ([1, 1, 1, 2, 3, 4], (1000, 3)),
        ([2, 2, 2, 4, 5, 6], (250, 4)),
        ([1, 5, 2, 2, 3, 4], (150, 2)),
        ([2, 2, 2, 2, 2, 2], (400, 6)),
        ([2, 2, 2, 3, 3, 3], (500, 6)),
        ([4, 4, 4, 6, 6, 6], (1000, 6)),
        ([5, 6, 5, 6, 6, 6], (700, 5)),
        ([1, 1, 1, 1, 1, 1], (2000, 6))
    ]

    resultados = map(
        lambda caso: (
            caso[0],
            caso[1],
            pontuacao_jogada(caso[0])
        ),
        casos
    )

    print(f"{'Entrada':<24} | {'Esperado':<12} | {'Obtido':<12} | Status")
    print("-" * 62)

    list(map(
        lambda r: print(
            f"{str(r[0]):<24} | "
            f"{str(r[1]):<12} | "
            f"{str(r[2]):<12} | "
            f"{'OK' if r[1] == r[2] else 'FALHOU'}"
        ),
        resultados
    ))


if __name__ == "__main__":
    print("Testes da pontuação:")
    testar_pontuacao()

    print("\nSimulação:")
    print(simular(1000))

### Validacao da pontuacao em Python

Os casos abaixo cobrem sequencia, jogadas sem pontuacao, trincas, duas trincas, dados individuais 1 e 5, e seis dados iguais.

In [ ]:
testar_pontuacao()

### Simulacao de 1000 partidas

A heuristica para quando o jogador para usa os pontos acumulados na rodada e a quantidade de dados restantes. Ela tende a parar ao atingir pelo menos 300 pontos ou quando restam poucos dados, reduzindo o risco de Farkle.

In [ ]:
simular(1000)

## Parte III - Kotlin

### 3.1 - Classificacao de senha

A funcao `passwordClassification(password: String): Pair<String, List<String>>` esta implementada no arquivo `kotlin/Farkle.kt`. Ela avalia cinco criterios: tamanho minimo, letra maiuscula, letra minuscula, digito e caractere especial.

```kotlin
// Exemplos esperados para passwordClassification:
// passwordClassification("abc")      -> (Fraca, [tamanho, maiuscula, digito, caractere especial])
// passwordClassification("Abc123")   -> (Moderada, [tamanho, caractere especial])
// passwordClassification("Abc123!@") -> (Muito Forte, [])
// passwordClassification("ABCDEFGH") -> (Moderada, [minuscula, digito, caractere especial])
// passwordClassification("Abcdefg1") -> (Forte, [caractere especial])
// passwordClassification("12345678") -> (Moderada, [maiuscula, minuscula, caractere especial])
```

### 3.2 - Traducao do Farkle para Kotlin

O codigo abaixo e a traducao funcional da parte Python para Kotlin, usando listas imutaveis, funcoes de alta ordem como `map`, `fold`, `filter` e `count`, e recursao com `tailrec` onde se aplica.

```kotlin
import kotlin.random.Random

fun rolarDados(n: Int): List<Int> =
    (1..n).map { Random.nextInt(1, 7) }

tailrec fun contar(lst: List<Int>, alvo: Int, acumulado: Int = 0): Int =
    when {
        lst.isEmpty() -> acumulado
        lst.first() == alvo -> contar(lst.drop(1), alvo, acumulado + 1)
        else -> contar(lst.drop(1), alvo, acumulado)
    }

fun valoresUnicos(lst: List<Int>): List<Int> =
    when {
        lst.isEmpty() -> emptyList()
        else -> {
            val primeiro = lst.first()
            val resto = valoresUnicos(lst.drop(1))
            if (primeiro in resto) resto else listOf(primeiro) + resto
        }
    }

fun ehSequencia(dados: List<Int>): Boolean =
    dados.sorted() == listOf(1, 2, 3, 4, 5, 6)

fun pontosPorNumero(dados: List<Int>, num: Int): Pair<Int, Int> {
    val qtd = contar(dados, num)

    if (qtd < 3) {
        return when (num) {
            1 -> Pair(qtd * 100, qtd)
            5 -> Pair(qtd * 50, qtd)
            else -> Pair(0, 0)
        }
    }

    val trincas = qtd / 3
    val resto = qtd % 3
    val pontosTrinca = if (num == 1) 1000 else num * 100
    val pontos = trincas * pontosTrinca
    val dadosUsados = trincas * 3

    return when (num) {
        1 -> Pair(pontos + resto * 100, dadosUsados + resto)
        5 -> Pair(pontos + resto * 50, dadosUsados + resto)
        else -> Pair(pontos, dadosUsados)
    }
}

fun pontuacaoJogada(dados: List<Int>): Pair<Int, Int> =
    if (ehSequencia(dados)) {
        Pair(1500, 6)
    } else {
        listOf(1, 2, 3, 4, 5, 6)
            .map { pontosPorNumero(dados, it) }
            .fold(Pair(0, 0)) { total, atual ->
                Pair(total.first + atual.first, total.second + atual.second)
            }
    }

fun escolhaAleatoria(nRestante: Int, pontos: Int): Boolean =
    listOf(true, false).random()

fun escolhaHeuristica(nRestante: Int, pontos: Int): Boolean =
    pontos < 300 && nRestante > 2

tailrec fun turno(
    pontosRodada: Int,
    nDados: Int,
    escolha: (Int, Int) -> Boolean,
    verbose: Boolean = true
): Int {
    val dados = rolarDados(nDados)
    val (pontos, dadosPontuadores) = pontuacaoJogada(dados)

    if (pontos == 0) {
        if (verbose) {
            println("Continua com $dados (0 pts) | FARKLE! Perdeu $pontosRodada pontos acumulados.")
        }
        return 0
    }

    val novoTotal = pontosRodada + pontos
    val nRestanteCalculado = nDados - dadosPontuadores
    val nRestante = if (nRestanteCalculado == 0) 6 else nRestanteCalculado

    if (verbose) {
        println("Lanca $dados ($pontos pts) | Acumulado: $novoTotal | Restantes: $nRestante dados")
    }

    return if (escolha(nRestante, novoTotal)) {
        turno(novoTotal, nRestante, escolha, verbose)
    } else {
        if (verbose) println("Jogador PARA com $novoTotal pontos")
        novoTotal
    }
}

tailrec fun partida(
    estrategia0: (Int, Int) -> Boolean,
    estrategia1: (Int, Int) -> Boolean,
    pontos: List<Int> = listOf(0, 0),
    jogador: Int = 0,
    pontuacaoMax: Int = 10000,
    verbose: Boolean = false
): Int? {
    val ponto0 = pontos[0]
    val ponto1 = pontos[1]

    if (ponto0 >= pontuacaoMax || ponto1 >= pontuacaoMax) {
        if (verbose) {
            println("\nFIM DE JOGO! Jogador 0: $ponto0 | Jogador 1: $ponto1")
        }

        return when {
            ponto0 >= pontuacaoMax && ponto1 >= pontuacaoMax && ponto0 > ponto1 -> 0
            ponto0 >= pontuacaoMax && ponto1 >= pontuacaoMax && ponto1 > ponto0 -> 1
            ponto0 >= pontuacaoMax && ponto1 >= pontuacaoMax -> null
            ponto0 >= pontuacaoMax -> 0
            else -> 1
        }
    }

    if (verbose) {
        println("\n- Jogador $jogador | Placar: $ponto0-$ponto1")
    }

    val escolha = if (jogador == 0) estrategia0 else estrategia1
    val pontosDoTurno = turno(0, 6, escolha, verbose)
    val novosPontos = listOf(
        ponto0 + if (jogador == 0) pontosDoTurno else 0,
        ponto1 + if (jogador == 1) pontosDoTurno else 0
    )

    return partida(estrategia0, estrategia1, novosPontos, 1 - jogador, pontuacaoMax, verbose)
}

fun passwordClassification(password: String): Pair<String, List<String>> {
    val criterios = listOf(
        Pair("tamanho", password.length >= 8),
        Pair("maiuscula", password.any { it in 'A'..'Z' }),
        Pair("minuscula", password.any { it in 'a'..'z' }),
        Pair("digito", password.any { it in '0'..'9' }),
        Pair("caractere especial", password.any { it in "!@#$%^&*()-+" })
    )

    val pontos = criterios.count { it.second }
    val naoAtendidos = criterios
        .filter { !it.second }
        .map { it.first }

    val classificacao = when (pontos) {
        0, 1 -> "Fraca"
        2, 3 -> "Moderada"
        4 -> "Forte"
        else -> "Muito Forte"
    }

    return Pair(classificacao, naoAtendidos)
}

fun simular(n: Int = 1000): String {
    val vitorias = (1..n)
        .map { partida(::escolhaHeuristica, ::escolhaAleatoria) }
        .count { it == 0 }
    val percentual = vitorias.toDouble() / n * 100.0

    return "Heuristica venceu %.2f%% das partidas contra aleatorio.".format(percentual)
}

fun demoFarkle() {
    println(passwordClassification("abc"))
    println(passwordClassification("Abc123"))
    println(passwordClassification("Abc123!@"))
    println(simular(1000))
}
```

### 3.3 - Validacao em Kotlin

O programa de testes abaixo valida `pontuacaoJogada` com mais de 10 casos. Ele mostra entrada, saida esperada, saida obtida, quantidade de casos que passaram e quantidade de falhas. Para falhas, tambem indica uma razao provavel e como corrigir.

```kotlin
data class CasoPontuacao(
    val dados: List<Int>,
    val esperado: Pair<Int, Int>
)

fun explicarFalha(obtido: Pair<Int, Int>, esperado: Pair<Int, Int>): String =
    when {
        obtido.first != esperado.first && obtido.second != esperado.second ->
            "pontuacao e quantidade de dados pontuadores diferentes; revisar regras de trincas, sequencia e dados individuais"
        obtido.first != esperado.first ->
            "pontuacao diferente; revisar soma das combinacoes pontuadoras"
        else ->
            "quantidade de dados pontuadores diferente; revisar contagem dos dados usados na pontuacao"
    }

fun main() {
    val casos = listOf(
        CasoPontuacao(listOf(1, 5, 2, 3, 4, 6), Pair(1500, 6)),
        CasoPontuacao(listOf(2, 2, 3, 4, 4, 6), Pair(0, 0)),
        CasoPontuacao(listOf(1, 1, 1, 2, 3, 4), Pair(1000, 3)),
        CasoPontuacao(listOf(2, 2, 2, 4, 5, 6), Pair(250, 4)),
        CasoPontuacao(listOf(1, 5, 2, 2, 3, 4), Pair(150, 2)),
        CasoPontuacao(listOf(2, 2, 2, 2, 2, 2), Pair(400, 6)),
        CasoPontuacao(listOf(2, 2, 2, 3, 3, 3), Pair(500, 6)),
        CasoPontuacao(listOf(4, 4, 4, 6, 6, 6), Pair(1000, 6)),
        CasoPontuacao(listOf(5, 6, 5, 6, 6, 6), Pair(700, 5)),
        CasoPontuacao(listOf(1, 1, 1, 1, 1, 1), Pair(2000, 6)),
        CasoPontuacao(listOf(5, 5, 5, 5, 5, 5), Pair(1000, 6)),
        CasoPontuacao(listOf(3, 3, 3, 1, 5, 2), Pair(450, 5))
    )

    val resultados = casos.map { caso ->
        val obtido = pontuacaoJogada(caso.dados)
        Triple(caso, obtido, obtido == caso.esperado)
    }

    println("%-24s | %-12s | %-12s | Status".format("Entrada", "Esperado", "Obtido"))
    println("-".repeat(62))

    resultados.forEach { (caso, obtido, passou) ->
        println(
            "%-24s | %-12s | %-12s | %s".format(
                caso.dados.toString(),
                caso.esperado.toString(),
                obtido.toString(),
                if (passou) "OK" else "FALHOU"
            )
        )
        if (!passou) {
            println("Razao provavel: ${explicarFalha(obtido, caso.esperado)}")
            println("Correcao: ajustar pontuacaoJogada/pontosPorNumero para refletir exatamente a tabela de pontuacao.")
        }
    }

    val passaram = resultados.count { it.third }
    val falharam = resultados.size - passaram

    println("Casos que passaram: $passaram")
    println("Casos que falharam: $falharam")
}
```

## 3.4 - Comparacao Kotlin vs Python Funcional

**Simplicidade:** Python costuma ser mais simples para escrever rapidamente, pois exige pouca declaracao de tipos e tem sintaxe curta. Kotlin exige mais estrutura, mas ainda permite codigo conciso com funcoes de expressao unica, lambdas e inferencia de tipos. No Farkle, Python ficou mais direto para prototipar, enquanto Kotlin deixou contratos mais explicitos.

**Ortogonalidade:** Kotlin apresenta uma combinacao mais regular entre funcoes, tipos, colecoes e lambdas. Python tambem e flexivel, mas mistura estilos funcional, imperativo e orientado a objetos com menos restricoes. Para programacao funcional, Kotlin orienta melhor o uso consistente de colecoes imutaveis e transformacoes.

**Tipos de Dados:** Python usa tipagem dinamica, entao listas, tuplas e funcoes sao manipuladas com muita liberdade. Kotlin usa tipagem estatica, com `List<Int>`, `Pair<Int, Int>` e assinaturas claras. Isso torna Kotlin mais verboso, mas reduz ambiguidades em funcoes como `pontuacaoJogada`.

**Projeto de Sintaxe:** Python valoriza indentacao e leitura proxima de pseudocodigo. Kotlin usa chaves e sintaxe parecida com linguagens da familia Java, mas adiciona recursos modernos como `when`, lambdas e funcoes de extensao. Em codigo funcional, Kotlin fica expressivo, mas Python ainda e mais leve visualmente.

**Suporte a Abstracao:** As duas linguagens permitem funcoes de alta ordem. Python tem `map`, `filter`, `reduce` e lambdas, embora `lambda` seja limitada a uma expressao. Kotlin oferece lambdas mais ricas, tipos funcionais e APIs de colecao muito integradas, o que favorece abstracoes funcionais mais robustas.

**Expressividade:** Python e muito expressivo para listas, recursao simples e experimentacao. Kotlin e expressivo quando se usa `map`, `fold`, `when`, destructuring e funcoes de expressao unica. No trabalho, Python foi melhor para implementar rapidamente; Kotlin foi melhor para documentar a intencao por meio dos tipos.

**Checagem de Tipos:** Python detecta muitos erros apenas em tempo de execucao. Kotlin detecta erros de tipo em compilacao, como retornar uma estrutura errada ou passar uma funcao com assinatura diferente. Isso aumenta a confiabilidade da traducao, principalmente nos testes de `pontuacaoJogada`.

**Manipulacao de Excecoes:** Python usa excecoes dinamicas e simples de disparar/capturar. Kotlin tambem usa excecoes, mas incentiva o uso de tipos nulos controlados e verificacoes mais explicitas. Para este projeto, quase nao foi necessario tratar excecoes, pois as funcoes trabalham com entradas bem definidas.

**Restricao de Aliases:** Python permite aliasing com facilidade; listas podem ser compartilhadas e modificadas se o programador nao tomar cuidado. Kotlin diferencia `List` e `MutableList`, ajudando a restringir alteracoes acidentais. Como o enunciado pede evitar mutacao, Kotlin oferece uma protecao conceitual melhor nesse ponto.

**Legibilidade:** Python e altamente legivel pela sintaxe enxuta. Kotlin tambem e legivel, mas os tipos e chaves acrescentam mais texto. Em compensacao, as assinaturas Kotlin ajudam a entender rapidamente o que cada funcao recebe e retorna.

**Escrita:** Python exige menos codigo para chegar a uma solucao funcional. Kotlin demanda mais cuidado inicial com tipos e estrutura, mas possui recursos de colecao muito bons. Para estudantes, Python facilita explorar a logica; Kotlin ajuda a consolidar a solucao de forma mais disciplinada.

**Confiabilidade:** Kotlin tende a ser mais confiavel em projetos maiores por causa da checagem estatica, nulidade controlada e listas imutaveis por padrao quando usamos `List`. Python e confiavel quando bem testado, mas depende mais da disciplina do programador. Neste trabalho, os testes sao essenciais para garantir que a versao Kotlin preserva o comportamento da versao Python.